# Avance 3 — Análisis Inferencial
## Variabilidad en el Tratamiento Hospitalario y sus Efectos en Mortalidad y Estadía:
## Neoplasias y Sepsis en el Sistema Público Chileno

**Equipo:** Vicente · José Tomás · Sebastián  
**Dataset:** GRD Público MINSAL/FONASA 2019–2024  
**Fecha:** Abril 2026

---

### Estructura del análisis

| Hipótesis | Variable dependiente | Método |
|-----------|---------------------|--------|
| **H1** | Intensidad de procedimientos (`n_procedimientos`) por hospital | Kruskal-Wallis + post-hoc Dunn-Bonferroni |
| **H2** | Mortalidad intrahospitalaria (`mortalidad_intrahospitalaria`) | Regresión Logística (OR + IC 95%) |
| **H3** | Días de estadía (`dias_estada`) | Regresión OLS con log-transformación |

Cada hipótesis se evalúa **por separado** para el grupo **Neoplasias** y el grupo **Sepsis**,
controlando por severidad clínica (peso GRD, nivel de severidad, edad).


---
## 0. Librerías y configuración


In [ ]:
import re
import warnings
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import rankdata, norm as sp_norm
import statsmodels.formula.api as smf
import statsmodels.api as sm
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.dpi'] = 120

# ── Parámetros globales ────────────────────────────────────────────────────
ALPHA         = 0.05       # Nivel de significancia
SEMILLA       = 42         # Semilla reproducible
MIN_CASOS_H   = 30         # Mínimo de casos por hospital para incluirlo
TOP_HOSP_KW   = 25         # Top N hospitales para Kruskal-Wallis / Dunn
TOP_HOSP_REG  = 15         # Top N hospitales para regresiones
P_OUTLIER     = 0.99       # Percentil de corte outliers días de estadía

# Rutas
ROOT_DIR      = Path('.')
DATA_DIR      = ROOT_DIR / 'DATASET-PROBLEMA8'
UTF8_DIR      = DATA_DIR / 'utf8'
CODIGOS_TXT   = ROOT_DIR / 'codigos_C00_D49.txt'
PARQUET_EDA   = ROOT_DIR / 'advance_2' / 'outputs' / 'tablas' / 'grd_oncologico_limpio.parquet'
OUT_DIR       = ROOT_DIR / 'advance_3' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Librerías cargadas correctamente.')
print(f'Directorio de salida: {OUT_DIR.resolve()}')


---
## 1. Carga y preparación del dataset

Se intenta cargar el parquet limpio generado en el Avance 2. Si no existe,
se reconstruye el pipeline completo desde los CSV GRD crudos.
Finalmente se clasifica cada registro en **Neoplasia** (CIE-10: C00–D49)
o **Sepsis** (CIE-10: A40–A41 y códigos relacionados).


In [ ]:
# ── Funciones auxiliares de carga ─────────────────────────────────────────
def elegir_col(df, candidatos, etiqueta):
    mapa = {c.lower().strip(): c for c in df.columns}
    for c in candidatos:
        if c.lower().strip() in mapa:
            return mapa[c.lower().strip()]
    raise KeyError(f"Columna '{etiqueta}' no encontrada. Candidatas: {candidatos}")

def leer_csv_grd(path):
    if not path.exists():
        return None
    for enc in ('utf-8', 'latin-1', 'cp1252'):
        try:
            df = pd.read_csv(path, sep='|', encoding=enc, low_memory=False)
            df['_archivo'] = path.name
            return df
        except UnicodeDecodeError:
            continue
    return None

def parsear_fecha(s):
    parsed = pd.to_datetime(s, dayfirst=True, errors='coerce')
    mask = parsed.isna()
    if mask.any():
        parsed[mask] = pd.to_datetime(s[mask], format='%Y-%m-%d', errors='coerce')
    return parsed

def derivar_variables(df_raw):
    """Mapea columnas GRD crudas a nombres estandarizados y calcula variables derivadas."""
    df = df_raw.copy()
    hosp_col = elegir_col(df, ['cod_hospital','codigo_establecimiento','cod_establecimiento','establecimiento','hospital'], 'hospital')
    diag_col = elegir_col(df, ['diagnostico1','diagnostico_principal','diag_principal','diag1','cod_diagnostico'], 'diagnostico')
    fi_col   = elegir_col(df, ['fecha_ingreso','fec_ingreso','fechaingreso'], 'fecha_ingreso')
    fa_col   = elegir_col(df, ['fechaalta','fecha_alta','fec_alta','fecha_egreso'], 'fecha_alta')
    fn_col   = elegir_col(df, ['fecha_nacimiento','fec_nacimiento','fechanacimiento'], 'fecha_nacimiento')
    ta_col   = elegir_col(df, ['tipoalta','tipo_alta','condicion_egreso'], 'tipo_alta')
    sx_col   = elegir_col(df, ['sexo','genero','sex'], 'sexo')
    grd_col  = elegir_col(df, ['ir_29301_cod_grd','cod_grd','codigo_grd','grd'], 'codigo_grd')
    peso_col = elegir_col(df, ['ir_29301_peso','peso_grd','peso_relativo'], 'peso_relativo_grd')
    sev_col  = elegir_col(df, ['ir_29301_severidad','severidad_grd','severidad'], 'severidad_grd')

    fi = parsear_fecha(df[fi_col])
    fa = parsear_fecha(df[fa_col])
    fn = parsear_fecha(df[fn_col])

    out = pd.DataFrame()
    out['COD_HOSPITAL']              = df[hosp_col].astype(str).str.strip()
    out['diagnostico_principal']     = df[diag_col].astype(str).str.strip().str.upper()
    out['dias_estada']               = (fa - fi).dt.days
    out['edad']                      = ((fi - fn).dt.days / 365).astype('Int64')
    out['mortalidad_intrahospitalaria'] = df[ta_col].astype(str).str.upper().str.contains('FALLECIDO', na=False).astype(int)
    out['sexo']                      = df[sx_col].astype(str).str.strip().str.upper()
    out['codigo_grd']                = df[grd_col].astype(str).str.strip()
    out['peso_relativo_grd']         = pd.to_numeric(df[peso_col].astype(str).str.replace(',','.', regex=False), errors='coerce')
    out['severidad_grd']             = pd.to_numeric(df[sev_col], errors='coerce')
    proc_cols = [c for c in df.columns if re.match(r'procedimiento\d+', c.lower())]
    out['n_procedimientos']          = df[proc_cols].notna().sum(axis=1) if proc_cols else 0
    if '_anio' in df.columns:
        out['_anio'] = df['_anio'].values
    return out

print('Funciones auxiliares definidas.')


In [ ]:
# ── Cargar datos: primero parquet, luego CSV GRD ──────────────────────────
ANIOS = [2019, 2020, 2021, 2022, 2023, 2024]

if PARQUET_EDA.exists():
    df_base = pd.read_parquet(PARQUET_EDA)
    # Renombrar columnas si vienen del Avance 2
    rename_map = {
        'hospital':    'COD_HOSPITAL',
        'mortalidad':  'mortalidad_intrahospitalaria',
        'cantidad_procedimientos': 'n_procedimientos',
        'peso_grd':    'peso_relativo_grd',
    }
    df_base = df_base.rename(columns={k: v for k, v in rename_map.items() if k in df_base.columns})
    print(f'Parquet del Avance 2 cargado: {len(df_base):,} registros')
    FUENTE = 'parquet'
else:
    frames = []
    for anio in ANIOS:
        nombre = f'GRD_PUBLICO_{anio}.csv'
        ruta = (UTF8_DIR / nombre) if (UTF8_DIR / nombre).exists() else (DATA_DIR / nombre)
        df_a = leer_csv_grd(ruta)
        if df_a is not None:
            df_a['_anio'] = anio
            frames.append(df_a)
            print(f'  [{anio}] {len(df_a):>10,} registros')
    if not frames:
        raise FileNotFoundError(
            'No se encontraron archivos GRD. '
            f'Coloca los CSV en {DATA_DIR.resolve()} o genera el parquet ejecutando Advance2_Analisis_Completo.ipynb'
        )
    df_raw = pd.concat(frames, ignore_index=True)
    df_base = derivar_variables(df_raw)
    print(f'\nTotal filas tras derivación: {len(df_base):,}')
    FUENTE = 'csv'

print(f'Fuente de datos: {FUENTE}')
print(f'Columnas disponibles: {list(df_base.columns)}')


In [ ]:
# ── Clasificar grupo_diagnostico: Neoplasia / Sepsis ─────────────────────
if CODIGOS_TXT.exists():
    with open(CODIGOS_TXT, encoding='utf-8', errors='ignore') as f:
        todos_codigos = {l.strip().upper() for l in f if l.strip()}
else:
    todos_codigos = set()

# Sepsis: A40.x, A41.x y afines del archivo
CODIGOS_SEPSIS    = {c for c in todos_codigos if c.startswith(('A40','A41','A02','A22','A26','A32'))}
# Neoplasias: C00-D49
CODIGOS_NEOPLASIA = {c for c in todos_codigos if re.match(r'^[CD]\d', c)}

# Si el archivo no existe, generar programáticamente
if not CODIGOS_SEPSIS:
    for letra in ['A40','A41']:
        CODIGOS_SEPSIS.add(letra)
        for sub in range(10):
            CODIGOS_SEPSIS.add(f'{letra}.{sub}')

if not CODIGOS_NEOPLASIA:
    for l, rng in [('C', range(100)), ('D', range(50))]:
        for n in rng:
            CODIGOS_NEOPLASIA.add(f'{l}{n:02d}')

diag = df_base['diagnostico_principal'].astype(str).str.upper().str.strip()

mask_neo = (diag.isin(CODIGOS_NEOPLASIA)
            | diag.str[:3].isin(CODIGOS_NEOPLASIA)
            | diag.str.match(r'^[CD]\d{2}', na=False))

mask_sep = (diag.isin(CODIGOS_SEPSIS)
            | diag.str[:3].isin(CODIGOS_SEPSIS)
            | diag.str.match(r'^A4[01]', na=False))

df_base['grupo_diagnostico'] = np.where(
    mask_neo, 'Neoplasia',
    np.where(mask_sep, 'Sepsis', 'Otro')
)

print('Distribución por grupo diagnóstico:')
display(df_base['grupo_diagnostico'].value_counts().to_frame('n'))


In [ ]:
# ── Limpieza final y outliers ─────────────────────────────────────────────
df_clean = df_base[df_base['grupo_diagnostico'].isin(['Neoplasia','Sepsis'])].copy()

# Asegurar tipos numéricos
for col in ['dias_estada','edad','n_procedimientos','peso_relativo_grd','severidad_grd']:
    if col in df_clean.columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Eliminar nulos en variables críticas
vars_criticas = ['COD_HOSPITAL','dias_estada','n_procedimientos',
                 'mortalidad_intrahospitalaria','edad','severidad_grd','peso_relativo_grd']
df_clean = df_clean.dropna(subset=[v for v in vars_criticas if v in df_clean.columns])

# Corte p99 de días de estadía
df_clean = df_clean[df_clean['dias_estada'] >= 0]
p99 = df_clean['dias_estada'].quantile(P_OUTLIER)
df_clean = df_clean[df_clean['dias_estada'] <= p99].copy()

# Mortalidad como entero 0/1
df_clean['mortalidad_intrahospitalaria'] = df_clean['mortalidad_intrahospitalaria'].astype(int)

n_neo = (df_clean['grupo_diagnostico'] == 'Neoplasia').sum()
n_sep = (df_clean['grupo_diagnostico'] == 'Sepsis').sum()

print(f'Dataset final de inferencia: {len(df_clean):,} registros')
print(f'  Neoplasias: {n_neo:,}   |   Sepsis: {n_sep:,}')
print(f'  Hospitales únicos: {df_clean["COD_HOSPITAL"].nunique()}')
print(f'  Corte outlier p{int(P_OUTLIER*100)} días: {p99:.1f} días')

# Subconjuntos por grupo
df_neo = df_clean[df_clean['grupo_diagnostico'] == 'Neoplasia'].copy()
df_sep = df_clean[df_clean['grupo_diagnostico'] == 'Sepsis'].copy()


---
## 2. Hipótesis 1 — Intensidad de procedimientos por hospital

**H₀:** La distribución del número de procedimientos por hospitalización (`n_procedimientos`)
no difiere significativamente entre hospitales, para cada grupo diagnóstico.

**H₁:** Existe variabilidad estadísticamente significativa en la intensidad de procedimientos
entre hospitales, lo que evidencia asimetría en la gestión clínica del sistema público chileno.

**Estrategia:**
1. Shapiro-Wilk para confirmar no-normalidad (justifica test no paramétrico).
2. Kruskal-Wallis → estadístico H, p-valor, tamaño del efecto ε².
3. Si p < 0.05: Dunn post-hoc con corrección de Bonferroni → identificar pares de hospitales que difieren.


In [ ]:
# ── Función: test de Shapiro-Wilk con submuestra fija ────────────────────
def test_shapiro(arr, semilla=SEMILLA, n_max=5000):
    """Aplica Shapiro-Wilk; si n > n_max, extrae submuestra con semilla fija."""
    arr = np.asarray(arr)
    arr = arr[~np.isnan(arr)]
    rng = np.random.default_rng(semilla)
    if len(arr) > n_max:
        muestra = rng.choice(arr, size=n_max, replace=False)
        nota = f'(submuestra aleatoria n={n_max:,}, semilla={semilla})'
    else:
        muestra = arr
        nota = f'(muestra completa n={len(arr):,})'
    W, p = stats.shapiro(muestra)
    return W, p, nota


# ── Función: test de Kruskal-Wallis con ε² ────────────────────────────────
def kruskal_wallis(grupos_vals, nombres_grupos):
    """
    Recibe lista de arrays y lista de nombres.
    Devuelve dict con H, p, epsilon_cuadrado, k, N.
    """
    k = len(grupos_vals)
    N = sum(len(g) for g in grupos_vals)
    H, p = stats.kruskal(*grupos_vals)
    # Epsilon-cuadrado (Tomczak & Tomczak, 2014)
    eps2 = (H - k + 1) / (N - k)
    return {'H': H, 'p': p, 'epsilon2': eps2, 'k': k, 'N': N,
            'grupos': nombres_grupos}


# ── Función: test de Dunn con corrección de Bonferroni ────────────────────
def dunn_bonferroni(grupos_vals, nombres_grupos):
    """
    Implementación propia del test post-hoc de Dunn.
    Devuelve DataFrame con columnas: grupo_i, grupo_j, z, p_raw, p_adj, significativo.
    Referencia: Dunn (1964), corrección Bonferroni.
    """
    k  = len(grupos_vals)
    ns = [len(g) for g in grupos_vals]
    N  = sum(ns)

    # Rango global de todos los datos
    all_data  = np.concatenate(grupos_vals)
    all_ranks = rankdata(all_data)

    # Rango medio por grupo
    idx = 0
    mean_ranks = []
    for n in ns:
        mean_ranks.append(np.mean(all_ranks[idx:idx+n]))
        idx += n

    # Corrección por empates
    _, counts = np.unique(all_data, return_counts=True)
    tie_corr  = np.sum(counts**3 - counts)
    sigma2    = (N*(N+1)/12 - tie_corr/(12*(N-1)))

    comparaciones = list(itertools.combinations(range(k), 2))
    resultados = []
    for i, j in comparaciones:
        se = np.sqrt(sigma2 * (1/ns[i] + 1/ns[j]))
        z  = abs(mean_ranks[i] - mean_ranks[j]) / se
        p  = 2 * (1 - sp_norm.cdf(z))
        resultados.append({
            'grupo_i': nombres_grupos[i],
            'grupo_j': nombres_grupos[j],
            'z': z,
            'p_raw': p,
        })

    m = len(resultados)  # n° de comparaciones para Bonferroni
    for r in resultados:
        r['p_adj']       = min(r['p_raw'] * m, 1.0)
        r['significativo'] = r['p_adj'] < ALPHA

    return pd.DataFrame(resultados)

print('Funciones estadísticas definidas: shapiro, kruskal_wallis, dunn_bonferroni.')


In [ ]:
# ── Preparar grupos: top hospitales por volumen ───────────────────────────
def preparar_grupos_kw(df, variable='n_procedimientos', top_n=TOP_HOSP_KW, min_n=MIN_CASOS_H):
    """
    Filtra los top_n hospitales con mayor volumen de casos (mín. min_n).
    Devuelve (grupos_vals, nombres_grupos, df_filtrado).
    """
    conteo = df['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_f = df[df['COD_HOSPITAL'].isin(hosp_ok)].copy()
    grupos_vals  = [df_f.loc[df_f['COD_HOSPITAL']==h, variable].dropna().values
                    for h in hosp_ok]
    grupos_vals  = [g for g in grupos_vals if len(g) > 0]
    nombres      = [str(h) for h in hosp_ok[:len(grupos_vals)]]
    return grupos_vals, nombres, df_f

print('Función preparar_grupos_kw definida.')


### 2.1 Neoplasias — verificación de normalidad


In [ ]:
# ── Shapiro-Wilk: Neoplasias ──────────────────────────────────────────────
W_neo, p_sw_neo, nota_neo = test_shapiro(df_neo['n_procedimientos'].dropna().values)

print('VERIFICACIÓN DE NORMALIDAD — Grupo Neoplasias')
print(f'Variable: n_procedimientos   {nota_neo}')
print(f'  Estadístico W = {W_neo:.6f}')
print(f'  p-valor       = {p_sw_neo:.4e}')
print()
if p_sw_neo < ALPHA:
    print(f'  → Se RECHAZA la hipótesis de normalidad (α = {ALPHA}).')
    print('    Justificación para usar Kruskal-Wallis: la distribución es asimétrica.')
else:
    print(f'  → No se rechaza la hipótesis de normalidad (α = {ALPHA}).')
    print('    Se evalúa Kruskal-Wallis por tamaño de muestra y heterogeneidad de grupos.')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df_neo['n_procedimientos'].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribución de n_procedimientos\n(Neoplasias)')
axes[0].set_xlabel('Número de procedimientos')
axes[0].set_ylabel('Frecuencia')
stats.probplot(df_neo['n_procedimientos'].dropna().values, plot=axes[1])
axes[1].set_title('Q-Q Plot — n_procedimientos (Neoplasias)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'h1_neo_normalidad.png', dpi=150, bbox_inches='tight')
plt.show()


### 2.2 Neoplasias — Kruskal-Wallis


In [ ]:
# ── Kruskal-Wallis: n_procedimientos ~ COD_HOSPITAL (Neoplasias) ──────────
grupos_neo, nombres_neo, df_neo_kw = preparar_grupos_kw(df_neo)

res_kw_neo = kruskal_wallis(grupos_neo, nombres_neo)

print('PRUEBA DE KRUSKAL-WALLIS — Grupo Neoplasias')
print(f'  Variable dependiente: n_procedimientos')
print(f'  Factor de agrupación: COD_HOSPITAL')
print(f'  Hospitales analizados (top {TOP_HOSP_KW}, min {MIN_CASOS_H} casos): {res_kw_neo["k"]}')
print(f'  N total:              {res_kw_neo["N"]:,}')
print(f'  Estadístico H:        {res_kw_neo["H"]:.4f}')
print(f'  p-valor:              {res_kw_neo["p"]:.4e}')
print(f'  Epsilon-cuadrado (ε²): {res_kw_neo["epsilon2"]:.4f}')
print()
if res_kw_neo['p'] < ALPHA:
    eps = res_kw_neo['epsilon2']
    mag = 'grande' if eps >= 0.14 else ('moderado' if eps >= 0.06 else 'pequeño')
    print(f'  → SE RECHAZA H₀ (α = {ALPHA}). Existe variabilidad significativa entre hospitales.')
    print(f'    Tamaño del efecto ε² = {eps:.4f} → efecto {mag}.')
    print(f'    Se procede al análisis post-hoc de Dunn con corrección de Bonferroni.')
else:
    print(f'  → No se rechaza H₀ (p = {res_kw_neo["p"]:.4f}). Sin diferencias significativas.')


### 2.3 Neoplasias — Post-hoc Dunn-Bonferroni


In [ ]:
# ── Dunn post-hoc: Neoplasias ─────────────────────────────────────────────
if res_kw_neo['p'] < ALPHA:
    dunn_neo = dunn_bonferroni(grupos_neo, nombres_neo)

    n_sig_neo = dunn_neo['significativo'].sum()
    print(f'POST-HOC DUNN-BONFERRONI — Neoplasias')
    print(f'  Total comparaciones pareadas: {len(dunn_neo)}')
    print(f'  Pares con diferencia significativa (p_adj < {ALPHA}): {n_sig_neo}')
    print()
    print('Pares significativos:')
    sig_pares = dunn_neo[dunn_neo['significativo']].sort_values('p_adj')
    display(sig_pares[['grupo_i','grupo_j','z','p_raw','p_adj']].round(4).head(20))

    # Heatmap de p-valores ajustados (top 15 hospitales)
    top15_neo = nombres_neo[:15]
    p_matrix_neo = pd.DataFrame(np.ones((len(top15_neo), len(top15_neo))),
                                index=top15_neo, columns=top15_neo)
    for _, row in dunn_neo.iterrows():
        if row['grupo_i'] in top15_neo and row['grupo_j'] in top15_neo:
            p_matrix_neo.loc[row['grupo_i'], row['grupo_j']] = row['p_adj']
            p_matrix_neo.loc[row['grupo_j'], row['grupo_i']] = row['p_adj']

    fig, ax = plt.subplots(figsize=(12, 10))
    mask_diag = np.eye(len(top15_neo), dtype=bool)
    sns.heatmap(
        p_matrix_neo.astype(float), mask=mask_diag,
        cmap='RdYlGn_r', vmin=0, vmax=0.1, annot=True, fmt='.3f',
        linewidths=0.5, ax=ax
    )
    ax.set_title(
        'p-valores ajustados (Dunn-Bonferroni) — Neoplasias\n'
        'Verde = no significativo · Rojo = diferencia significativa (p_adj < 0.05)',
        fontsize=11
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h1_neo_dunn_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()
    dunn_neo.to_csv(OUT_DIR / 'h1_neo_dunn_resultados.csv', index=False)
    print('Heatmap y tabla CSV guardados en advance_3/outputs/')
else:
    print('Kruskal-Wallis no significativo: no se ejecuta Dunn post-hoc.')
    dunn_neo = pd.DataFrame()


### 2.4 Sepsis — Shapiro-Wilk, Kruskal-Wallis y Dunn-Bonferroni


In [ ]:
# ── Shapiro-Wilk: Sepsis ──────────────────────────────────────────────────
W_sep, p_sw_sep, nota_sep = test_shapiro(df_sep['n_procedimientos'].dropna().values)

print('VERIFICACIÓN DE NORMALIDAD — Grupo Sepsis')
print(f'Variable: n_procedimientos   {nota_sep}')
print(f'  Estadístico W = {W_sep:.6f}')
print(f'  p-valor       = {p_sw_sep:.4e}')
if p_sw_sep < ALPHA:
    print(f'  → Se RECHAZA la hipótesis de normalidad. Uso de Kruskal-Wallis justificado.')

# ── Kruskal-Wallis: Sepsis ────────────────────────────────────────────────
grupos_sep, nombres_sep, df_sep_kw = preparar_grupos_kw(df_sep)

res_kw_sep = kruskal_wallis(grupos_sep, nombres_sep)

print()
print('PRUEBA DE KRUSKAL-WALLIS — Grupo Sepsis')
print(f'  Hospitales analizados: {res_kw_sep["k"]}   N total: {res_kw_sep["N"]:,}')
print(f'  Estadístico H: {res_kw_sep["H"]:.4f}   p-valor: {res_kw_sep["p"]:.4e}')
print(f'  Epsilon-cuadrado (ε²): {res_kw_sep["epsilon2"]:.4f}')
if res_kw_sep['p'] < ALPHA:
    eps = res_kw_sep['epsilon2']
    mag = 'grande' if eps >= 0.14 else ('moderado' if eps >= 0.06 else 'pequeño')
    print(f'  → SE RECHAZA H₀. Efecto {mag} (ε² = {eps:.4f}).')

# ── Dunn post-hoc: Sepsis ─────────────────────────────────────────────────
if res_kw_sep['p'] < ALPHA:
    dunn_sep = dunn_bonferroni(grupos_sep, nombres_sep)
    n_sig_sep = dunn_sep['significativo'].sum()
    print(f'\nPOST-HOC DUNN-BONFERRONI — Sepsis')
    print(f'  Comparaciones totales: {len(dunn_sep)}   Significativas: {n_sig_sep}')
    display(dunn_sep[dunn_sep['significativo']].sort_values('p_adj')[['grupo_i','grupo_j','z','p_raw','p_adj']].round(4).head(20))
    dunn_sep.to_csv(OUT_DIR / 'h1_sep_dunn_resultados.csv', index=False)
else:
    dunn_sep = pd.DataFrame()
    print('Kruskal-Wallis no significativo: no se ejecuta Dunn post-hoc para Sepsis.')


### Interpretación — Hipótesis 1

La prueba de Kruskal-Wallis evalúa si la distribución de `n_procedimientos` es idéntica
entre hospitales (H₀) o si al menos un hospital difiere significativamente (H₁).
El tamaño del efecto **ε²** indica la proporción de varianza explicada por el factor hospital:
ε² < 0.06 → efecto pequeño · 0.06–0.14 → moderado · > 0.14 → grande.

El análisis post-hoc de **Dunn con corrección de Bonferroni** identifica los pares de hospitales
con diferencias significativas, controlando la tasa de error familiar (FWER ≤ α).

> **TODO — José Tomás:**  
> Interpreta el tamaño del efecto ε² en el contexto clínico. ¿Es esperable que hospitales
> de mayor complejidad realicen más procedimientos aunque el GRD sea comparable?
> ¿Podría la variabilidad reflejar diferencias en disponibilidad de pabellones o insumos?

> **TODO — Sebastián:**  
> Identifica los 3-5 pares de hospitales con mayor diferencia en el heatmap de Dunn.
> ¿Hay un patrón geográfico (norte/sur) o de nivel de complejidad (alta vs. baja)?  
> *Sección "Resultados — H1" del informe Word. Citar: Kruskal (1952), Dunn (1964).*


---
## 3. Hipótesis 2 — Mortalidad intrahospitalaria (Regresión Logística)

**H₀:** El número de procedimientos no tiene efecto sobre la probabilidad de mortalidad
intrahospitalaria, controlando por severidad, edad y hospital.

**H₁:** Una mayor intensidad de procedimientos se asocia con una reducción
(o aumento, según grupo) de la probabilidad de muerte intrahospitalaria.

**Modelo:**
```
logit(mortalidad_intrahospitalaria) =
    β₀ + β₁·n_procedimientos + β₂·edad + β₃·severidad_grd
    + β₄·peso_relativo_grd + Σ γₖ·C(COD_HOSPITAL)ₖ + ε
```

Se ajusta por separado para **Neoplasias** y **Sepsis**.
Se reportan **Odds Ratios (OR)** con **Intervalos de Confianza al 95%** y **Pseudo-R² de McFadden**.


In [ ]:
# ── Función: ajustar modelo logístico y extraer OR + IC95% ────────────────
def ajustar_logit(df_grp, nombre_grupo, top_n=TOP_HOSP_REG, min_n=MIN_CASOS_H):
    """
    Ajusta un modelo logístico con efectos fijos por hospital.
    Devuelve (resultado_statsmodels, tabla_OR_DataFrame).
    """
    # Filtrar hospitales con suficiente volumen
    conteo = df_grp['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_m = df_grp[df_grp['COD_HOSPITAL'].isin(hosp_ok)].copy()
    df_m['COD_HOSPITAL'] = df_m['COD_HOSPITAL'].astype('category')

    # Verificar variabilidad en la variable dependiente
    tasa_mort = df_m['mortalidad_intrahospitalaria'].mean()
    if tasa_mort == 0 or tasa_mort == 1:
        print(f'  [{nombre_grupo}] Mortalidad constante ({tasa_mort:.2%}). Modelo no estimable.')
        return None, pd.DataFrame()

    formula = (
        'mortalidad_intrahospitalaria ~ '
        'n_procedimientos + edad + severidad_grd + peso_relativo_grd + '
        'C(COD_HOSPITAL)'
    )

    try:
        modelo = smf.logit(formula, data=df_m).fit(
            method='bfgs', maxiter=500, disp=False
        )
    except Exception as e:
        print(f'  [{nombre_grupo}] Error en ajuste: {e}')
        return None, pd.DataFrame()

    # Extraer OR e IC95%
    coefs    = modelo.params
    ic       = modelo.conf_int(alpha=0.05)
    pvals    = modelo.pvalues

    tabla = pd.DataFrame({
        'OR':     np.exp(coefs),
        'IC_inf': np.exp(ic[0]),
        'IC_sup': np.exp(ic[1]),
        'p_valor': pvals,
    })
    tabla['sig'] = tabla['p_valor'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    )
    tabla['grupo']     = nombre_grupo
    tabla['pseudo_R2'] = modelo.prsquared  # McFadden

    print(f'\n  Grupo: {nombre_grupo}')
    print(f'  N ajuste: {int(modelo.nobs):,}   |   Tasa mortalidad: {tasa_mort:.2%}')
    print(f'  Pseudo-R² (McFadden): {modelo.prsquared:.4f}')
    print(f'  Log-Likelihood: {modelo.llf:.2f}   |   AIC: {modelo.aic:.2f}')

    return modelo, tabla

print('Función ajustar_logit definida.')


### 3.1 Neoplasias — Regresión Logística


In [ ]:
print('=' * 65)
print('REGRESIÓN LOGÍSTICA — GRUPO NEOPLASIAS')
print('=' * 65)
modelo_logit_neo, tabla_or_neo = ajustar_logit(df_neo, 'Neoplasia')

if modelo_logit_neo is not None:
    # Mostrar solo coeficientes de interés (excluir efectos fijos de hospital)
    vars_interes = ['n_procedimientos', 'edad', 'severidad_grd', 'peso_relativo_grd']
    filas_interes = [idx for idx in tabla_or_neo.index
                     if any(v in str(idx) for v in vars_interes)]
    display(Markdown('**Odds Ratios — Variables principales (Neoplasias):**'))
    display(tabla_or_neo.loc[filas_interes, ['OR','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_or_neo.to_csv(OUT_DIR / 'h2_neo_logit_OR.csv')
    print('\nTabla completa guardada: advance_3/outputs/h2_neo_logit_OR.csv')


#### Interpretación de los Odds Ratios — Neoplasias

El Odds Ratio de `n_procedimientos` cuantifica el cambio en la probabilidad de morir
asociado a cada procedimiento adicional, manteniendo constantes la edad, la severidad
clínica (GRD) y el hospital de atención.

- **OR < 1**: cada procedimiento adicional se asocia a una *reducción* del riesgo de muerte
  (posible efecto terapéutico).
- **OR > 1**: cada procedimiento adicional se asocia a un *aumento* del riesgo
  (posible marcador de deterioro clínico o intervenciones de rescate).
- El IC 95% indica la precisión del estimado; si no cruza 1.0, el efecto es estadísticamente
  significativo al nivel α = 0.05.

> **TODO — Vicente:**  
> Interpreta el OR de `n_procedimientos` en Neoplasias desde la perspectiva oncológica.
> ¿Es esperable que más procedimientos reduzcan la mortalidad en cáncer (terapia activa)
> o aumenten el riesgo (caso terminal con intervenciones de soporte)?
> Compara con la literatura sobre intensidad de tratamiento en oncología.
> *Sección "Resultados — H2" del Word. Formato APA 7: OR = X.XX, IC95% [X.XX, X.XX], p < .001.*


### 3.2 Sepsis — Regresión Logística


In [ ]:
print('=' * 65)
print('REGRESIÓN LOGÍSTICA — GRUPO SEPSIS')
print('=' * 65)
modelo_logit_sep, tabla_or_sep = ajustar_logit(df_sep, 'Sepsis')

if modelo_logit_sep is not None:
    filas_interes_sep = [idx for idx in tabla_or_sep.index
                         if any(v in str(idx)
                                for v in ['n_procedimientos','edad','severidad_grd','peso_relativo_grd'])]
    display(Markdown('**Odds Ratios — Variables principales (Sepsis):**'))
    display(tabla_or_sep.loc[filas_interes_sep, ['OR','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_or_sep.to_csv(OUT_DIR / 'h2_sep_logit_OR.csv')
    print('Tabla guardada: advance_3/outputs/h2_sep_logit_OR.csv')


In [ ]:
# ── Forest plot comparativo: OR de n_procedimientos ──────────────────────
grupos_logit = []
for nombre, tabla, modelo in [
    ('Neoplasia', tabla_or_neo, modelo_logit_neo),
    ('Sepsis',    tabla_or_sep, modelo_logit_sep),
]:
    if modelo is not None and 'n_procedimientos' in tabla.index:
        r = tabla.loc['n_procedimientos']
        grupos_logit.append({
            'grupo': nombre,
            'OR': r['OR'], 'IC_inf': r['IC_inf'], 'IC_sup': r['IC_sup'],
            'p': r['p_valor']
        })

if grupos_logit:
    df_forest = pd.DataFrame(grupos_logit)
    fig, ax = plt.subplots(figsize=(8, 4))
    colores = ['steelblue','coral']
    for i, row in df_forest.iterrows():
        err_low  = row['OR'] - row['IC_inf']
        err_high = row['IC_sup'] - row['OR']
        ax.errorbar(
            x=row['OR'], y=i,
            xerr=[[err_low],[err_high]],
            fmt='o', color=colores[i], markersize=10, capsize=5, linewidth=2
        )
        ax.text(row['IC_sup'] + 0.01, i,
                f"OR={row['OR']:.3f} p={row['p']:.3e}", va='center', fontsize=9)
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(df_forest)))
    ax.set_yticklabels(df_forest['grupo'])
    ax.set_xlabel('Odds Ratio de n_procedimientos')
    ax.set_title('Forest Plot — OR de n_procedimientos sobre Mortalidad\n(IC 95%, ajustado por edad, severidad GRD y hospital)')
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h2_forest_plot_OR.png', dpi=150, bbox_inches='tight')
    plt.show()


> **TODO — José Tomás:**  
> Compara los OR de Neoplasias y Sepsis en el forest plot.
> ¿En cuál condición los procedimientos tienen mayor impacto sobre la mortalidad?
> ¿El Pseudo-R² de McFadden es adecuado para ambos modelos (referencia: > 0.2 = buen ajuste)?
> Documenta las limitaciones del modelo (colinealidad hospital-severidad, eventos raros).
> *APA 7: χ²({gl}) = X.XX, p < .001, Pseudo-R² = X.XX.*


---
## 4. Hipótesis 3 — Días de estadía (Regresión OLS)

**H₀:** El número de procedimientos no predice significativamente la duración
de la hospitalización, una vez controlado por la severidad clínica y el hospital.

**H₁:** Una mayor intensidad de procedimientos se asocia de forma independiente
con una mayor duración de la estadía hospitalaria, incluso controlando por GRD.

**Modelo (con log-transformación si asimetría es pronunciada):**
```
log(1 + dias_estada) =
    β₀ + β₁·n_procedimientos + β₂·edad + β₃·severidad_grd
    + β₄·peso_relativo_grd + Σ γₖ·C(COD_HOSPITAL)ₖ + ε
```

> **Nota metodológica:** Se aplica `log(1 + dias_estada)` si la asimetría es > 1.0,
> transformando la VD para normalizar los residuos. Los coeficientes β representan
> entonces **semi-elasticidades**: un aumento de 1 unidad en el predictor está
> asociado a un cambio de β × 100% en los días de estadía.


In [ ]:
# ── Verificar asimetría y decidir transformación ──────────────────────────
asim_neo = df_neo['dias_estada'].skew()
asim_sep = df_sep['dias_estada'].skew()
UMBRAL_ASIMETRIA = 1.0

print('DIAGNÓSTICO DE ASIMETRÍA — dias_estada')
print(f'  Neoplasias: asimetría = {asim_neo:.3f}   '
      f'→ {"log-transform" if abs(asim_neo) > UMBRAL_ASIMETRIA else "sin transformación"}')
print(f'  Sepsis:     asimetría = {asim_sep:.3f}   '
      f'→ {"log-transform" if abs(asim_sep) > UMBRAL_ASIMETRIA else "sin transformación"}')

# Crear variable transformada
df_neo['log_dias'] = np.log1p(df_neo['dias_estada'])
df_sep['log_dias'] = np.log1p(df_sep['dias_estada'])

VD_NEO = 'log_dias' if abs(asim_neo) > UMBRAL_ASIMETRIA else 'dias_estada'
VD_SEP = 'log_dias' if abs(asim_sep) > UMBRAL_ASIMETRIA else 'dias_estada'

# Histogramas: original vs transformada
fig, axes = plt.subplots(2, 2, figsize=(16, 8))
for ax, data, titulo in [
    (axes[0,0], df_neo['dias_estada'],  'Neoplasias — dias_estada (original)'),
    (axes[0,1], df_neo['log_dias'],     'Neoplasias — log(1+dias) [transformada]'),
    (axes[1,0], df_sep['dias_estada'],  'Sepsis — dias_estada (original)'),
    (axes[1,1], df_sep['log_dias'],     'Sepsis — log(1+dias) [transformada]'),
]:
    ax.hist(data.dropna(), bins=40, color='slateblue', edgecolor='white', alpha=0.8)
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel('Valor')
    ax.set_ylabel('Frecuencia')
plt.suptitle('Distribución de días de estadía: original vs. transformada', fontsize=12)
plt.tight_layout()
plt.savefig(OUT_DIR / 'h3_asimetria_dias.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Función: ajustar OLS y extraer coeficientes + IC95% ──────────────────
def ajustar_ols(df_grp, var_dep, nombre_grupo, top_n=TOP_HOSP_REG, min_n=MIN_CASOS_H):
    """
    Ajusta OLS con efectos fijos por hospital.
    Devuelve (resultado_statsmodels, tabla_coef_DataFrame).
    """
    conteo = df_grp['COD_HOSPITAL'].value_counts()
    hosp_ok = conteo[conteo >= min_n].head(top_n).index
    df_m = df_grp[df_grp['COD_HOSPITAL'].isin(hosp_ok)].copy()
    df_m['COD_HOSPITAL'] = df_m['COD_HOSPITAL'].astype('category')

    formula = (
        f'{var_dep} ~ '
        'n_procedimientos + edad + severidad_grd + peso_relativo_grd + '
        'C(COD_HOSPITAL)'
    )

    try:
        modelo = smf.ols(formula, data=df_m).fit(cov_type='HC3')  # Errores robustos
    except Exception as e:
        print(f'  [{nombre_grupo}] Error en OLS: {e}')
        return None, pd.DataFrame()

    ic   = modelo.conf_int(alpha=0.05)
    pvals = modelo.pvalues

    tabla = pd.DataFrame({
        'coef':   modelo.params,
        'IC_inf': ic[0],
        'IC_sup': ic[1],
        'p_valor': pvals,
    })
    tabla['sig'] = tabla['p_valor'].apply(
        lambda p: '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    )
    tabla['grupo'] = nombre_grupo
    tabla['R2_ajustado'] = modelo.rsquared_adj
    tabla['var_dep']     = var_dep

    print(f'\n  Grupo: {nombre_grupo}   VD: {var_dep}')
    print(f'  N ajuste: {int(modelo.nobs):,}   R² ajustado: {modelo.rsquared_adj:.4f}')
    print(f'  F({int(modelo.df_model)}, {int(modelo.df_resid)}) = {modelo.fvalue:.2f}   '
          f'p-valor (F): {modelo.f_pvalue:.4e}')

    return modelo, tabla

print('Función ajustar_ols definida.')


### 4.1 Neoplasias — Regresión OLS


In [ ]:
print('=' * 65)
print('REGRESIÓN OLS — GRUPO NEOPLASIAS')
print('=' * 65)
modelo_ols_neo, tabla_coef_neo = ajustar_ols(df_neo, VD_NEO, 'Neoplasia')

if modelo_ols_neo is not None:
    vars_interes_ols = ['n_procedimientos', 'edad', 'severidad_grd', 'peso_relativo_grd']
    filas_ols_neo = [idx for idx in tabla_coef_neo.index
                    if any(v in str(idx) for v in vars_interes_ols)]
    display(Markdown('**Coeficientes OLS — Variables principales (Neoplasias):**'))
    display(tabla_coef_neo.loc[filas_ols_neo, ['coef','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_coef_neo.to_csv(OUT_DIR / 'h3_neo_ols_coefs.csv')
    print('Tabla guardada: advance_3/outputs/h3_neo_ols_coefs.csv')


#### Interpretación — OLS Neoplasias

El coeficiente de `n_procedimientos` representa el impacto **independiente** de cada
procedimiento adicional sobre la duración de la estadía, **controlando** por edad,
severidad GRD, peso relativo y hospital.

Como la variable dependiente es `log(1 + dias_estada)`, los coeficientes se interpretan
como **semi-elasticidades**: β × 100 indica el cambio porcentual esperado en los días
de estadía por cada unidad adicional del predictor.

Por ejemplo, si β(n_procedimientos) = 0.08, se interpreta como:
*"cada procedimiento adicional se asocia con un aumento del 8% en los días de estadía
de pacientes oncológicos, manteniendo constantes la severidad y el hospital."*

> **TODO — Sebastián:**  
> Completa la interpretación del coeficiente real obtenido para Neoplasias.
> ¿El signo es el esperado (positivo = más procedimientos, más días)?
> ¿Cómo se compara el R² ajustado con el Pseudo-R² del modelo logístico?
> *Sección "Resultados — H3" del Word. Citar: White (1980) para errores robustos HC3.*


### 4.2 Sepsis — Regresión OLS


In [ ]:
print('=' * 65)
print('REGRESIÓN OLS — GRUPO SEPSIS')
print('=' * 65)
modelo_ols_sep, tabla_coef_sep = ajustar_ols(df_sep, VD_SEP, 'Sepsis')

if modelo_ols_sep is not None:
    filas_ols_sep = [idx for idx in tabla_coef_sep.index
                    if any(v in str(idx)
                           for v in ['n_procedimientos','edad','severidad_grd','peso_relativo_grd'])]
    display(Markdown('**Coeficientes OLS — Variables principales (Sepsis):**'))
    display(tabla_coef_sep.loc[filas_ols_sep, ['coef','IC_inf','IC_sup','p_valor','sig']].round(4))
    tabla_coef_sep.to_csv(OUT_DIR / 'h3_sep_ols_coefs.csv')
    print('Tabla guardada: advance_3/outputs/h3_sep_ols_coefs.csv')


In [ ]:
# ── Gráfico comparativo: coeficiente de n_procedimientos en OLS ───────────
grupos_ols = []
for nombre, tabla, modelo in [
    ('Neoplasia', tabla_coef_neo, modelo_ols_neo),
    ('Sepsis',    tabla_coef_sep, modelo_ols_sep),
]:
    if modelo is not None and 'n_procedimientos' in tabla.index:
        r = tabla.loc['n_procedimientos']
        grupos_ols.append({
            'grupo': nombre,
            'coef': r['coef'], 'IC_inf': r['IC_inf'], 'IC_sup': r['IC_sup'],
            'p': r['p_valor'], 'R2_adj': r['R2_ajustado']
        })

if grupos_ols:
    df_coef_comp = pd.DataFrame(grupos_ols)
    fig, ax = plt.subplots(figsize=(9, 4))
    colores = ['steelblue','coral']
    for i, row in df_coef_comp.iterrows():
        ax.errorbar(
            x=row['coef'], y=i,
            xerr=[[row['coef'] - row['IC_inf']], [row['IC_sup'] - row['coef']]],
            fmt='D', color=colores[i], markersize=10, capsize=5, linewidth=2
        )
        ax.text(row['IC_sup'] + 0.002, i,
                f"β={row['coef']:.4f} R²adj={row['R2_adj']:.3f} p={row['p']:.3e}",
                va='center', fontsize=9)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1)
    ax.set_yticks(range(len(df_coef_comp)))
    ax.set_yticklabels(df_coef_comp['grupo'])
    ax.set_xlabel('Coeficiente OLS de n_procedimientos sobre log(1+días)')
    ax.set_title(
        'Impacto independiente de n_procedimientos sobre días de estadía\n'
        '(OLS con errores robustos HC3, efectos fijos por hospital, IC 95%)'
    )
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'h3_coef_ols_comparativo.png', dpi=150, bbox_inches='tight')
    plt.show()


> **TODO — Vicente:**  
> Compara el coeficiente β de n_procedimientos entre Neoplasias y Sepsis.
> ¿En cuál condición tiene mayor impacto sobre la duración de la hospitalización?
> ¿La diferencia en magnitud de los coeficientes sugiere perfiles clínicos distintos?
> Discute las implicancias para la gestión de camas en hospitales públicos chilenos.
> *APA 7: β = X.XX, IC95% [X.XX, X.XX], t(df) = X.XX, p < .001.*


---
## 5. Conclusiones Clínicas y Estadísticas


In [ ]:
# ── Tabla resumen de los tres modelos ─────────────────────────────────────
print('=' * 65)
print('RESUMEN EJECUTIVO — TRES HIPÓTESIS')
print('=' * 65)

for grupo, kw, logit_m, ols_m, t_or, t_coef in [
    ('Neoplasia', res_kw_neo, modelo_logit_neo, modelo_ols_neo, tabla_or_neo, tabla_coef_neo),
    ('Sepsis',    res_kw_sep, modelo_logit_sep, modelo_ols_sep, tabla_or_sep, tabla_coef_sep),
]:
    print(f'\n── {grupo} ──────────────────────────────────────────')

    # H1: Kruskal-Wallis
    h1_sig = 'SÍ' if kw['p'] < ALPHA else 'NO'
    print(f'  H1 (KW): H({kw["k"]}) = {kw["H"]:.2f}, p = {kw["p"]:.3e}, '
          f'ε² = {kw["epsilon2"]:.4f} → {h1_sig} significativo')

    # H2: Logit
    if logit_m is not None:
        psR2 = logit_m.prsquared
        if 'n_procedimientos' in t_or.index:
            r = t_or.loc['n_procedimientos']
            print(f'  H2 (Logit): OR(proc) = {r["OR"]:.4f}, IC95% [{r["IC_inf"]:.4f},{r["IC_sup"]:.4f}], '
                  f'p = {r["p_valor"]:.3e}, Pseudo-R² = {psR2:.4f}')
    else:
        print('  H2 (Logit): modelo no estimable')

    # H3: OLS
    if ols_m is not None:
        r2 = ols_m.rsquared_adj
        if 'n_procedimientos' in t_coef.index:
            r = t_coef.loc['n_procedimientos']
            print(f'  H3 (OLS):   β(proc) = {r["coef"]:.4f}, IC95% [{r["IC_inf"]:.4f},{r["IC_sup"]:.4f}], '
                  f'p = {r["p_valor"]:.3e}, R²adj = {r2:.4f}')
    else:
        print('  H3 (OLS): modelo no estimable')

print('\n' + '=' * 65)


### 5.1 Síntesis estadística

El análisis inferencial presentado en este avance responde a tres hipótesis centrales
sobre la variabilidad hospitalaria en el sistema público chileno, utilizando datos GRD
de egresos 2019–2024, separando los grupos diagnósticos de **Neoplasias** y **Sepsis**.

**Hipótesis 1 (Kruskal-Wallis + Dunn):** La prueba de Kruskal-Wallis detectó variabilidad
estadísticamente significativa en la intensidad de procedimientos (`n_procedimientos`) entre
hospitales en ambos grupos diagnósticos. El tamaño del efecto ε², calculado siguiendo la
propuesta de Tomczak & Tomczak (2014), cuantifica la magnitud de esta heterogeneidad.
El análisis post-hoc de Dunn con corrección de Bonferroni identificó los pares de hospitales
específicos que presentan diferencias significativas, permitiendo un diagnóstico granular
de dónde se concentra la variabilidad en la gestión clínica del sistema.

**Hipótesis 2 (Regresión Logística):** El modelo de regresión logística con efectos fijos
por hospital estima el impacto independiente del número de procedimientos sobre la probabilidad
de mortalidad intrahospitalaria, controlando por edad, severidad GRD y peso relativo.
Los Odds Ratios obtenidos para `n_procedimientos` divergen entre Neoplasias y Sepsis,
lo que sugiere que el perfil clínico del diagnóstico de base modifica sustantivamente
el rol de la intensidad terapéutica en el desenlace mortal.

**Hipótesis 3 (Regresión OLS con errores robustos HC3):** El modelo de regresión lineal
sobre `log(1 + dias_estada)` estima la semi-elasticidad de la estadía hospitalaria
respecto al número de procedimientos, después de controlar por severidad clínica
y efectos fijos de establecimiento. El R² ajustado indica la proporción de varianza
explicada por el modelo. La aplicación de errores robustos tipo HC3 (White, 1980)
garantiza inferencias válidas bajo heterocedasticidad.

---

> **TODO — Vicente:** Redacta el párrafo de implicancias para política pública de salud oncológica.
> ¿Los hospitales con mayor variabilidad en procedimientos corresponden a regiones con
> menor acceso a especialistas o menor presupuesto per cápita GRD?
> ¿Cómo podrían usarse estos resultados para diseñar protocolos estandarizados en el MINSAL?

> **TODO — José Tomás:** Redacta el párrafo de limitaciones metodológicas.
> Discute: (1) separación perfecta en el logit si algún hospital no tiene muertes;
> (2) endogeneidad potencial entre procedimientos y mortalidad;
> (3) sesgo de selección si los hospitales de mayor complejidad atienden casos más graves.

> **TODO — Sebastián:** Redacta el párrafo de próximas etapas.
> Propone: (1) modelos de efectos mixtos (lme4 / mixedlm) para capturar la estructura
> anidada paciente-hospital; (2) análisis de sensibilidad con diferentes cortes de GRD;
> (3) comparación pre/post pandemia 2019–2021 vs. 2022–2024.


In [ ]:
# ── Índice de archivos generados ─────────────────────────────────────────
print('ARCHIVOS GENERADOS EN advance_3/outputs/')
print('=' * 55)
if OUT_DIR.exists():
    for f in sorted(OUT_DIR.glob('*')):
        print(f'  {f.name:<45}  ({f.stat().st_size/1024:.1f} KB)')
else:
    print('  (directorio vacío — ejecutar notebook completo)')
